# City of Boston Subconscious Getting Started

This notebook walks you through your first API calls with **Subconscious**, a long-horizon reasoning model — and builds up to connecting an agent to the **City of Boston open-data MCP server** for live city data.

Subconscious speaks the **OpenAI Chat Completions** protocol, so you use the official `openai` SDK pointed at Subconscious's base URL.

Two steps to start:

1. Paste your API key below (get one at [subconscious.dev/platform](https://www.subconscious.dev/platform))
2. Click "Run All", or run each cell one by one.

Let's go!

In [ ]:
# You can find your API key on the platform:
# https://www.subconscious.dev/platform

# **Paste your API key here**
api_key = "sk-***"

In [ ]:
# Subconscious is OpenAI-compatible. We also use the `mcp` SDK later to talk to
# the Boston open-data server, and `pydantic` for structured output.
!pip install openai mcp pydantic > /dev/null 2>&1

In [ ]:
# Point the OpenAI SDK at Subconscious. That's the whole setup.
from openai import OpenAI

client = OpenAI(base_url="https://api.subconscious.dev/v1", api_key=api_key)

# There is one model today:
MODEL = "subconscious/tim-qwen3.6-27b"

## *You're in.* Let's try it out!

The simplest thing you can do is send a message and read the reply — exactly like the OpenAI API.

In [ ]:
completion = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What are some fun facts about the city of Boston?"}],
)

print(completion.choices[0].message.content)

## Structured Output with Pydantic

Sometimes you want **structured data** instead of free text. Pass a JSON schema via `response_format` and the reply is guaranteed to match it; we validate it into [Pydantic](https://docs.pydantic.dev/) models so the fields are typed.

In [ ]:
from pydantic import BaseModel


class BostonNeighborhood(BaseModel):
    name: str
    known_for: str
    population_estimate: str


class BostonNeighborhoods(BaseModel):
    neighborhoods: list[BostonNeighborhood]


completion = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": "List 5 Boston neighborhoods with what each is known for and a rough population estimate.",
    }],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "BostonNeighborhoods",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "neighborhoods": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "known_for": {"type": "string"},
                                "population_estimate": {"type": "string"},
                            },
                            "required": ["name", "known_for", "population_estimate"],
                            "additionalProperties": False,
                        },
                    },
                },
                "required": ["neighborhoods"],
                "additionalProperties": False,
            },
        },
    },
)

result = BostonNeighborhoods.model_validate_json(completion.choices[0].message.content)
for n in result.neighborhoods:
    print(f"{n.name:20s} | {n.known_for:40s} | Pop: {n.population_estimate}")

## Connecting to Boston Open Data via MCP (client-side)

Now let's connect the agent to a **real data source**. The City of Boston runs an [MCP server](https://data-mcp.boston.gov/mcp) exposing its open-data portal — no API key or extra signup.

We connect to the MCP server, turn its tools into standard **OpenAI function tools**, and hand them to the model via `tools=`. When the model calls one, we forward it to the MCP server and feed the result back — looping until the model has an answer. Tools run **client-side**; Subconscious just decides which to call.

(This is the same pattern the [`hack-cli-starter`](https://github.com/subconscious-systems/subconscious/tree/main/examples/hack-cli-starter) example uses to build a full terminal agent.)

In [ ]:
import json
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

BOSTON_MCP_URL = "https://data-mcp.boston.gov/mcp"


def _tool_text(result) -> str:
    """Flatten an MCP tool result into plain text for the model."""
    parts = [getattr(b, "text", None) or str(b) for b in result.content]
    return "\n".join(parts) if parts else "(no content)"


def _mcp_tools_to_openai(mcp_tools):
    """Convert MCP tool definitions into OpenAI function-tool specs."""
    return [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema or {"type": "object", "properties": {}},
            },
        }
        for t in mcp_tools
    ]


async def run_boston_agent(question: str, max_steps: int = 8, verbose: bool = True) -> str:
    """Let the model use the Boston MCP tools as native OpenAI function tools."""
    async with streamablehttp_client(BOSTON_MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            mcp_tools = (await session.list_tools()).tools
            tools = _mcp_tools_to_openai(mcp_tools)
            tool_names = {t.name for t in mcp_tools}

            messages = [
                {
                    "role": "system",
                    "content": (
                        "You answer questions about the City of Boston using the available "
                        "tools. Call a tool when you need real data instead of guessing."
                    ),
                },
                {"role": "user", "content": question},
            ]

            for _ in range(max_steps):
                resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
                msg = resp.choices[0].message

                if not msg.tool_calls:
                    return msg.content or ""

                # Record the assistant turn that requested the tool(s).
                messages.append({
                    "role": "assistant",
                    "content": msg.content,
                    "tool_calls": [
                        {"id": tc.id, "type": "function",
                         "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                        for tc in msg.tool_calls
                    ],
                })

                # Forward each call to the MCP server and feed the result back.
                for tc in msg.tool_calls:
                    name = tc.function.name
                    try:
                        args = json.loads(tc.function.arguments)
                    except json.JSONDecodeError:
                        args = {}
                    if verbose:
                        print(f"[tool] {name}({args})")

                    if name not in tool_names:
                        observation = f"Error: unknown tool {name!r}"
                    else:
                        try:
                            observation = _tool_text(await session.call_tool(name, args))
                        except Exception as exc:  # tool errors aren't fatal — let the model recover
                            observation = f"Tool {name} failed: {exc}"

                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": observation})

            return "Stopped: reached the step limit without a final answer."

In [ ]:
# Jupyter and Colab allow top-level `await`.
answer = await run_boston_agent("How many restaurants are in Boston?")
print(answer)

## More complex interactions

The best instructions tell the model its goal, the pitfalls to watch for, and the output format you want. The Boston MCP exposes datasets for crime, population, emergency services, the bicycle network, utility data, women-owned businesses, and more.

In [ ]:
instructions = '''
Goal: Given a hypothetical $200M capital budget, recommend which
BPS buildings to renovate vs replace vs close, optimizing for
students served per dollar and remaining useful life.

Pitfalls:
- Renovation cost is not in the dataset. Estimate from building
  size and condition score, and state your assumptions.
- Don't ignore enrollment trends. Replacing a building losing
  students is different from one at capacity.
- "Educational suitability" can't always be fixed with money.
  Call out where it's a layout problem vs a maintenance problem.

Output Format:
- 3 ranked lists (renovate / replace / close) with total cost
- Short justification per school
- Stated assumptions at the top
'''

answer = await run_boston_agent(instructions, max_steps=12)
print(answer)

## What's Next?

You've covered the core building blocks: chat, structured output, and a client-side MCP tool loop against real Boston data. Ideas to explore next:

- **Build a real agent** — [`hack-cli-starter`](https://github.com/subconscious-systems/subconscious/tree/main/examples/hack-cli-starter) turns the loop above into a full terminal agent with multiple MCP servers.
- **Bring your own MCP** — point `run_boston_agent` at any HTTP MCP server URL.
- **Anthropic-compatible API** — Subconscious also speaks the Anthropic Messages API at `https://api.subconscious.dev`.

### Useful Links

| Resource | Link |
|----------|------|
| Documentation | [docs.subconscious.dev](https://docs.subconscious.dev) |
| API Reference | [docs.subconscious.dev/api-reference](https://docs.subconscious.dev/api-reference/introduction) |
| Platform | [subconscious.dev/platform](https://www.subconscious.dev/platform) |

Happy building!